> **Experimental alternative model:** Requires a locally obtained `../data/market_risk.xlsx` workbook (sheet `Data`). It is not included because market-data redistribution rights are unverified. Run from `notebooks/`. This model uses a separate methodology and event definitions from the CIR notebook; outputs are not directly comparable and are not credit ratings.

## CELL 01 — Imports

In [ ]:
# CELL 01 -- Imports & Seed
import os
import warnings, numpy as np, pandas as pd
os.makedirs('../figures', exist_ok=True)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from scipy.optimize import minimize
from scipy.stats import t as t_dist
warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_PATH = "../data/market_risk.xlsx"   # change path if needed
print("Ready.")


## CELL 02 — Load & Clean Data


In [ ]:
# CELL 02 -- Load & clean data
def load_data(path):
    df = pd.read_excel(path, sheet_name="Data")
    df["treasury_date"] = pd.to_datetime(df["treasury_date"], errors="coerce")
    df["date_cdx"]      = pd.to_datetime(df["date_cdx"],      errors="coerce")
    df["dbaa_date"]     = pd.to_datetime(df["dbaa_date"],      errors="coerce")

    treasury = (df[["treasury_date","5_year_treasurey"]]
                .dropna(subset=["treasury_date"])
                .rename(columns={"treasury_date":"date","5_year_treasurey":"rf_pct"})
                .sort_values("date").reset_index(drop=True))
    cdx = (df[["date_cdx","cdx_5y"]]
           .dropna(subset=["date_cdx"])
           .rename(columns={"date_cdx":"date","cdx_5y":"cdx_bps"})
           .sort_values("date").reset_index(drop=True))
    dbaa = (df[["dbaa_date","dbaa"]]
            .dropna(subset=["dbaa_date"])
            .rename(columns={"dbaa_date":"date","dbaa":"baa_yield_pct"})
            .sort_values("date").reset_index(drop=True))

    print(f"Treasury : {treasury['date'].min().date()} to {treasury['date'].max().date()} ({len(treasury):,})")
    print(f"CDX      : {cdx['date'].min().date()} to {cdx['date'].max().date()} ({len(cdx):,})")
    print(f"DBAA     : {dbaa['date'].min().date()} to {dbaa['date'].max().date()} ({len(dbaa):,})")
    return treasury, cdx, dbaa

treasury, cdx_raw, dbaa_raw = load_data(DATA_PATH)


## CELL 03 — Build Unified Spread Series


In [ ]:
# CELL 03 -- Build Unified Monthly Spread Series
# ─────────────────────────────────────────────────────────────────
# Convert each dataset to month-end first, then align them monthly.
# No spread forward-fill.
#
# Zones:
#   2007-2016 → BAA proxy = (DBAA - Treasury) × 0.45
#   2017-2019 → ignored gap
#   2020 onward → CDX actual
# ─────────────────────────────────────────────────────────────────

BAA_SCALAR = 0.45

def to_month_end(df, value_cols):
    out = df.copy().sort_values("date")
    out = (
        out.set_index("date")[value_cols]
           .resample("ME")
           .last()
           .reset_index()
    )
    return out

def build_spread_monthly(treasury, cdx_raw, dbaa_raw):
    # Convert each dataset to month-end first
    treasury_m = to_month_end(treasury, ["rf_pct"])
    cdx_m      = to_month_end(cdx_raw,  ["cdx_bps"])
    dbaa_m     = to_month_end(dbaa_raw, ["baa_yield_pct"])

    # Monthly alignment on Treasury month-end spine
    m = treasury_m.merge(cdx_m, on="date", how="left")
    m = m.merge(dbaa_m, on="date", how="left")

    # BAA proxy only for 2007-2016
    m["baa_proxy_bps"] = np.where(
        (m["date"] >= "2007-01-01") &
        (m["date"] <= "2016-12-31") &
        m["baa_yield_pct"].notna() &
        m["rf_pct"].notna(),
        (m["baa_yield_pct"] - m["rf_pct"]) * 100 * BAA_SCALAR,
        np.nan
    )

    # CDX actual only from 2020 onward
    m["cdx_actual_bps"] = np.where(
        m["date"] >= "2020-01-01",
        m["cdx_bps"],
        np.nan
    )

    # Unified spread: CDX actual first, then BAA proxy
    m["cdx_unified_bps"] = np.where(
        m["cdx_actual_bps"].notna(),
        m["cdx_actual_bps"],
        np.where(m["baa_proxy_bps"].notna(), m["baa_proxy_bps"], np.nan)
    )

    # Source label
    m["source"] = np.where(
        m["cdx_actual_bps"].notna(),
        "CDX_actual",
        np.where(m["baa_proxy_bps"].notna(), "BAA_proxy", "missing")
    )

    # Drop months with no real spread data
    m = m.dropna(subset=["cdx_unified_bps"]).reset_index(drop=True)

    return m

aligned_m = build_spread_monthly(treasury, cdx_raw, dbaa_raw)

# ── Two Clean Calibration Windows ────────────────────────────────

df_old_m = aligned_m[
    (aligned_m["date"] >= "2007-01-01") &
    (aligned_m["date"] <= "2016-12-31") &
    (aligned_m["source"] == "BAA_proxy")
].copy().reset_index(drop=True)

df_recent_m = aligned_m[
    (aligned_m["date"] >= "2020-01-01") &
    (aligned_m["source"] == "CDX_actual")
].copy().reset_index(drop=True)

# ── Starting Conditions ───────────────────────────────────────────

recent_cdx = aligned_m[aligned_m["source"] == "CDX_actual"].copy()

S0_BPS = recent_cdx.tail(3)["cdx_unified_bps"].median()
RF_PCT = recent_cdx.iloc[-1]["rf_pct"]

# ── Summary ───────────────────────────────────────────────────────

print(aligned_m["source"].value_counts().to_string())

print(f"\nOld window   : {len(df_old_m)} obs | "
      f"{df_old_m['cdx_unified_bps'].min():.0f}-"
      f"{df_old_m['cdx_unified_bps'].max():.0f} bps | "
      f"{df_old_m['date'].min().date()} to {df_old_m['date'].max().date()}")

print(f"Recent window: {len(df_recent_m)} obs | "
      f"{df_recent_m['cdx_unified_bps'].min():.0f}-"
      f"{df_recent_m['cdx_unified_bps'].max():.0f} bps | "
      f"{df_recent_m['date'].min().date()} to {df_recent_m['date'].max().date()}")

print(f"\nStarting: S0={S0_BPS:.1f} bps (3m median)  rf={RF_PCT:.2f}%")

In [ ]:
# CELL 03b -- Unified Spread Series 

baa = aligned_m[aligned_m["source"] == "BAA_proxy"]
cdx = aligned_m[aligned_m["source"] == "CDX_actual"]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(baa["date"], baa["cdx_unified_bps"],
        color="#D85A30", lw=1.8, label="BAA proxy — scaled  (2007–2016)")
ax.plot(cdx["date"], cdx["cdx_unified_bps"],
        color="#378ADD", lw=1.8, label="CDX actual (2020–2026)")
ax.fill_between(baa["date"], baa["cdx_unified_bps"], alpha=0.07, color="#D85A30")
ax.fill_between(cdx["date"], cdx["cdx_unified_bps"], alpha=0.07, color="#378ADD")

for val, color, ls, label in [
    (df_old_m["cdx_unified_bps"].mean(),    "#D85A30", "--",
     f'BAA mean   {df_old_m["cdx_unified_bps"].mean():.1f} bps'),
    (df_old_m["cdx_unified_bps"].median(),  "#993c1d", ":",
     f'BAA median {df_old_m["cdx_unified_bps"].median():.1f} bps'),
    (df_recent_m["cdx_unified_bps"].mean(),   "#378ADD", "--",
     f'CDX mean   {df_recent_m["cdx_unified_bps"].mean():.1f} bps'),
    (df_recent_m["cdx_unified_bps"].median(), "#0C447C", ":",
     f'CDX median {df_recent_m["cdx_unified_bps"].median():.1f} bps'),
]:
    ax.axhline(val, color=color, linestyle=ls, lw=1.2, alpha=0.85, label=label)

# Gap shading
ax.axvspan(pd.Timestamp("2017-01-01"), pd.Timestamp("2019-12-31"),
           alpha=0.05, color="gray")
ax.text(pd.Timestamp("2018-06-01"), ax.get_ylim()[1] * 0.92,
        "Calm period\nexcluded", fontsize=8, color="gray",
        ha="center", va="top")

# Event annotations
events = [
    ("2008-10-01", "GFC Peak",            "top"),
    ("2011-08-01", "US Debt Ceiling\n& S&P Downgrade", "top"),
    ("2016-02-01", "Oil / China\nStress", "top"),
    ("2020-03-01", "COVID-19\nPeak",      "top"),
    ("2022-10-01", "Fed Rate\nHikes",     "bottom"),
    ("2023-03-01", "SVB\nStress",         "bottom"),
]
for date_str, label, pos in events:
    dt      = pd.Timestamp(date_str)
    closest = aligned_m.iloc[(aligned_m["date"] - dt).abs().argsort()[:1]]
    y_val   = closest["cdx_unified_bps"].values[0]
    y_off   = 20 if pos == "top" else -25
    ax.annotate(label, xy=(dt, y_val), xytext=(dt, y_val + y_off),
                fontsize=7.5, color="#C0392B", ha="center",
                arrowprops=dict(arrowstyle="-", color="#C0392B", lw=0.8),
                bbox=dict(boxstyle="round,pad=0.25", fc="white",
                          alpha=0.75, ec="lightgray"))

# 70/30 blend note
ax.text(0.98, 0.05, "OU parameters blended 70% CDX / 30% BAA",
        transform=ax.transAxes, fontsize=8, color="gray",
        ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.6, ec="lightgray"))

ax.set_xlabel("Date")
ax.set_ylabel("Spread (bps)")
ax.set_title("Unified Spread Series — Model Input\n"
             "BAA proxy × 0.45 scalar (2007–2016)  +  CDX actual (2020–2026)",
             fontsize=11)
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.grid(True, alpha=0.15)
fig.tight_layout()
plt.show()


# ── Chart 2: US Treasury Rate ─────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(treasury["date"], treasury["rf_pct"],
        color="#2E7D32", lw=1.8, label="5Y US Treasury rate (historical)")
ax.fill_between(treasury["date"], treasury["rf_pct"],
                alpha=0.08, color="#2E7D32")

ax.axvspan(pd.Timestamp("2009-01-01"), pd.Timestamp("2015-12-01"),
           alpha=0.06, color="gray")
ax.text(pd.Timestamp("2012-01-01"), 0.3,
        "Near-zero rate\n(2009–2015)", fontsize=7.5, color="gray", ha="center")

ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2022-02-01"),
           alpha=0.06, color="gray")
ax.text(pd.Timestamp("2021-01-01"), 0.3,
        "Near-zero rate\n(2020–2022)", fontsize=7.5, color="gray", ha="center")

ax.axhline(RF_PCT, color="#E53935", ls="--", lw=2,
           label=f"Model rate (fixed) = {RF_PCT:.2f}%")

ax.set_xlabel("Date")
ax.set_ylabel("Rate (%)")
ax.set_title("5Y US Treasury Rate — Historical & Model Input", fontsize=12)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.15)
fig.tight_layout()
plt.show()


## CELL 04 — CPDO Parameters


In [ ]:
# CELL 04 -- CPDO Parameters (monthly timestep, 10-year product)

# ── Deal Structure ────────────────────────────────────────────────────────────
FACE_VALUE      = 1_000_000
DT              = 1/12              # monthly step
N_PERIODS       = 120               # 10yr x 12 months
N_NAMES         = 125               # actual CDX NA IG index composition
LGD             = 0.60              # Moody's IG historical average -- assumption
MATURITY        = 10.0

# ── Leverage Caps by Regime ───────────────────────────────────────────────────
LEV_CAP         = 15               

# ── Risk Controls ─────────────────────────────────────────────────────────────
SAFETY_BUFFER   = 0.15              # cash-out trigger: NAV < 15% of par (€150k)
SURPLUS_THRESH  = 0.05              # surplus lock: de-risk when all obligations covered
        

# ── Calibration Weights ───────────────────────────────────────────────────────
W_RECENT        = 0.70              # 2020-2026 (post-COVID market regime)
W_OLD           = 0.30              # 2007-2016 (includes GFC stress period)

# ── Fees ─────────────────────────────────────────────────────────────────────
ARRANGEMENT_FEE  = 0.01             # 1% upfront -- investor pays manager, day one
ADMIN_FEE_BPS    = 20               # 20bps/yr ongoing -- investor pays manager
FACILITY_FEE_BPS = 3.5              # 3.5bps/yr on notional -- manager pays dealer

# ── Monte Carlo ───────────────────────────────────────────────────────────────
N_PATHS = 10_000

# ── Derived Values ────────────────────────────────────────────────────────────
NAV0          = FACE_VALUE * (1 - ARRANGEMENT_FEE)    # starting NAV after upfront fee
CASHOUT_LEVEL = FACE_VALUE * SAFETY_BUFFER             # €150,000 -- fixed absolute floor on par
ADMIN_MONTH   = (ADMIN_FEE_BPS / 10_000) * FACE_VALUE * DT  # ~€167/month fixed

print(f"Timestep        : monthly (DT={DT:.4f})")
print(f"Periods         : {N_PERIODS} (10yr x 12)")
print(f"NAV at t=0      : ${NAV0:,.0f}")
print(f"Cash-out level  : ${CASHOUT_LEVEL:,.0f}  (fixed, based on par)")
print(f"Paths           : {N_PATHS:,}")
print(f"S0              : {S0_BPS:.1f} bps  (from Cell 03)")
print(f"rf              : {RF_PCT:.2f}%     (from Cell 03, held constant)")


## CELL 05 — Regime Detection


In [ ]:
# CELL 05 
# Classifies each month as normal (0) or crisis (1) using two signals:
# (1) 6-month rolling log-spread volatility
# (2) Absolute spread level
# Crisis = either signal above its 90th historical percentile

def detect_regimes(df, spread_col="cdx_unified_bps", date_col="date",
                   window=6, pct=0.90, max_gap_days=65):

    x = df[[date_col, spread_col]].dropna().copy()
    x[date_col] = pd.to_datetime(x[date_col])
    x = x.sort_values(date_col).reset_index(drop=True)

    # Identify breaks in monthly continuity
    x["gap"] = x[date_col].diff().dt.days > max_gap_days
    x["segment"] = x["gap"].cumsum()

    x["logS"] = np.log(np.maximum(x[spread_col], 1e-6))

    # Do not compute log changes across missing-data gaps
    x["dlog"] = x.groupby("segment")["logS"].diff()

    # Do not compute rolling volatility across missing-data gaps
    x["roll_vol"] = (
        x.groupby("segment")["dlog"]
         .transform(lambda s: s.rolling(window=window, min_periods=window).std())
    )

    x["lvl"] = x[spread_col]

    vol_thresh = x["roll_vol"].quantile(pct)
    lvl_thresh = x["lvl"].quantile(pct)

    x["is_crisis"] = (
        (x["roll_vol"] > vol_thresh) |
        (x["lvl"] > lvl_thresh)
    ).astype(int)

    # Optional: do not classify as crisis where rolling vol is unavailable
    x.loc[x["roll_vol"].isna(), "is_crisis"] = 0

    return x, vol_thresh, lvl_thresh   # ← this was removed by mistake


regime_df, vol_thresh, lvl_thresh = detect_regimes(aligned_m)

crisis_flags = regime_df["is_crisis"].values
roll_vol     = regime_df["roll_vol"].values

total    = len(crisis_flags)
n_crisis = crisis_flags.sum()
print(f"Crisis months: {n_crisis} / {total} ({100*n_crisis/total:.1f}%)")
print(f"Normal months: {total-n_crisis} / {total} ({100*(total-n_crisis)/total:.1f}%)")
print(f"Vol threshold: {vol_thresh:.5f}  (90th pct of 6-month rolling vol)")
print(f"Lvl threshold: {lvl_thresh:.1f} bps (90th pct of spread level)")


In [ ]:
# CELL 05b -- Regime Detection Validation
# Shows: (1) spread with crisis periods shaded
#         (2) rolling volatility vs threshold
#         (3) sensitivity to threshold choice

from matplotlib.patches import Patch
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(
    "Regime Detection Validation — 90th Percentile Threshold",
    fontsize=12,
    fontweight="bold"
)

dates = regime_df["date"].reset_index(drop=True)
spreads = regime_df["cdx_unified_bps"].reset_index(drop=True)
cf_plot = regime_df["is_crisis"].reset_index(drop=True)
roll_vol_plot = regime_df["roll_vol"].reset_index(drop=True)
gap_mask = regime_df["gap"].reset_index(drop=True)

# ── Panel 1: Spread with crisis shading ──────────────────────────
ax = axes[0]

spread_plot = spreads.copy()
spread_plot.loc[gap_mask] = np.nan

ax.plot(
    dates,
    spread_plot,
    color="#1976D2",
    lw=1.5,
    label="CDX Unified Spread"
)

# Missing-data gaps
for i in np.where(gap_mask)[0]:
    if i > 0:
        ax.axvspan(dates.iloc[i - 1], dates.iloc[i],
                   alpha=0.18, color="gray")

# Crisis shading, without bridging gaps
in_crisis = False
start_idx = None

for i, flag in enumerate(cf_plot):
    if gap_mask.iloc[i]:
        if in_crisis and start_idx is not None and i > 0:
            ax.axvspan(dates.iloc[start_idx], dates.iloc[i - 1],
                       alpha=0.25, color="red")
        in_crisis = False
        start_idx = None
        continue

    if flag == 1 and not in_crisis:
        in_crisis = True
        start_idx = i
    elif flag == 0 and in_crisis:
        ax.axvspan(dates.iloc[start_idx], dates.iloc[i],
                   alpha=0.25, color="red")
        in_crisis = False
        start_idx = None

if in_crisis and start_idx is not None:
    ax.axvspan(dates.iloc[start_idx], dates.iloc[len(cf_plot) - 1],
               alpha=0.25, color="red")

ax.set_ylabel("Spread (bps)")
ax.set_title("Spread Series with Crisis Periods; Missing Data Shown in Gray")
ax.grid(True, alpha=0.15)

crisis_patch = Patch(color="red", alpha=0.3, label="Crisis regime")
missing_patch = Patch(color="gray", alpha=0.18, label="Missing data")
ax.legend(handles=[ax.lines[0], crisis_patch, missing_patch], fontsize=8)

# ── Panel 2: Rolling volatility vs threshold ─────────────────────
ax = axes[1]

roll_vol_broken = roll_vol_plot.copy()
roll_vol_broken.loc[gap_mask] = np.nan

ax.plot(
    dates,
    roll_vol_broken,
    color="#E53935",
    lw=1.5,
    label="6-month rolling log-vol"
)

ax.axhline(
    vol_thresh,
    color="black",
    ls="--",
    lw=1.5,
    label=f"90th pct threshold = {vol_thresh:.4f}"
)

for i in np.where(gap_mask)[0]:
    if i > 0:
        ax.axvspan(dates.iloc[i - 1], dates.iloc[i],
                   alpha=0.18, color="gray")

valid_fill = roll_vol_broken.notna() & (roll_vol_broken > vol_thresh)

ax.fill_between(
    dates,
    roll_vol_broken,
    vol_thresh,
    where=valid_fill,
    alpha=0.3,
    color="red",
    label="Above threshold"
)

ax.set_ylabel("Log-spread volatility")
ax.set_title("Rolling Volatility Signal vs 90th Percentile Threshold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Panel 3: Threshold sensitivity ───────────────────────────────
ax = axes[2]

pct_values = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
crisis_pcts = []

for pct in pct_values:
    reg_test, _, _ = detect_regimes(aligned_m, pct=pct)
    crisis_pcts.append(reg_test["is_crisis"].mean() * 100)

ax.plot(
    [p * 100 for p in pct_values],
    crisis_pcts,
    color="#1976D2",
    marker="o",
    lw=2,
    markersize=7
)

ax.axvline(
    90,
    color="black",
    ls="--",
    lw=1.5,
    label="Chosen threshold (90th pct)"
)

ax.set_xlabel("Percentile threshold (%)")
ax.set_ylabel("% months classified as crisis")
ax.set_title("Sensitivity of Crisis Classification to Threshold Choice")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

idx_90 = pct_values.index(0.90)

ax.annotate(
    f"{crisis_pcts[idx_90]:.1f}% crisis\n(chosen)",
    xy=(90, crisis_pcts[idx_90]),
    xytext=(92, crisis_pcts[idx_90] + 5),
    fontsize=8,
    color="black",
    arrowprops=dict(arrowstyle="->", color="black", lw=0.8)
)

plt.tight_layout()
plt.savefig("../figures/experimental_regime_validation.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary statistics
print("Regime Classification Summary:")
print(f"  Total months     : {len(cf_plot)}")
print(f"  Crisis months    : {cf_plot.sum()} ({cf_plot.mean()*100:.1f}%)")
print(f"  Normal months    : {(1-cf_plot).sum()} ({(1-cf_plot).mean()*100:.1f}%)")
print(f"  Vol threshold    : {vol_thresh:.5f} (90th pct 6-month rolling log-vol)")
print(f"  Level threshold  : {lvl_thresh:.1f} bps (90th pct spread level)")
print(f"  Missing gaps     : {gap_mask.sum()}")

print("\n  Justification for 90th percentile:")
print(f"  - Conservative: catches more stress periods than 85th ({crisis_pcts[3]:.1f}%)")
print("  - Not overly restrictive: leaves enough normal periods for calibration")
print("  - Standard in regime-switching literature-style threshold diagnostics")

## CELL 06 — Markov Transition Matrix


In [ ]:
# CELL 06
# States: 0 = normal, 1 = crisis
# P[i,j] = P(next state = j | current state = i)
# Fitted by counting observed transitions in the crisis flag series
#
# Important:
# Skip transitions across missing-data gaps, e.g. last 2016 point -> first 2020 point.

def fit_markov_from_regime_df(regime_df):
    flags = regime_df["is_crisis"].astype(int).values
    gaps = regime_df["gap"].astype(bool).values

    counts = np.zeros((2, 2))

    for t in range(len(flags) - 1):
        # Skip transition into a new segment
        if gaps[t + 1]:
            continue

        counts[flags[t], flags[t + 1]] += 1

    row_sums = counts.sum(axis=1, keepdims=True)

    fallback_P = np.array(
        [[0.95, 0.05],
         [0.30, 0.70]],
        dtype=float
    )

    P = np.divide(
        counts,
        row_sums,
        out=fallback_P.copy(),
        where=row_sums != 0
    )

    return P, counts


# Fit Markov transition matrix using gap-aware regime dataframe
P_markov, counts = fit_markov_from_regime_df(regime_df)


print("Transition count matrix:")
print(f"  Normal -> Normal : {counts[0,0]:,.0f}")
print(f"  Normal -> Crisis : {counts[0,1]:,.0f}")
print(f"  Crisis -> Normal : {counts[1,0]:,.0f}")
print(f"  Crisis -> Crisis : {counts[1,1]:,.0f}")

print(f"\nTransition probability matrix P:")
print(f"  P[Normal->Normal] = {P_markov[0,0]:.5f}  P[Normal->Crisis] = {P_markov[0,1]:.5f}")
print(f"  P[Crisis->Normal] = {P_markov[1,0]:.5f}  P[Crisis->Crisis] = {P_markov[1,1]:.5f}")

# Stationary distribution, long-run fraction of time in each state
eigvals, eigvecs = np.linalg.eig(P_markov.T)
stat = np.real(eigvecs[:, np.argmin(np.abs(eigvals - 1))])
stat = stat / stat.sum()

print(f"\nStationary distribution: Normal={stat[0]:.3f}  Crisis={stat[1]:.3f}")

## CELL 07 — Log-OU Spread Calibration


In [ ]:
# CELL 07 -- Regime-Switching Log-OU Calibration (moment-matching)
# theta = -log(rho) / dt   where rho = lag-1 autocorrelation
# mu    = sample mean of log-spread
# sigma = std(dY) / sqrt(dt)

def fit_log_ou_regime(spread_series, crisis_flag_series, dt_obs, label=""):
    S     = spread_series.dropna().values / 10_000
    logS  = np.log(np.maximum(S, 1e-6))
    dlogS = np.diff(logS)
    logSt = logS[:-1]

    cf = crisis_flag_series
    if len(cf) > len(dlogS): cf = cf[-len(dlogS):]
    if len(cf) < len(dlogS): cf = np.concatenate([np.zeros(len(dlogS)-len(cf), dtype=int), cf])

    results = {}
    for state, name in [(0, "normal"), (1, "crisis")]:
        mask = (cf == state)
        dY   = dlogS[mask]
        Yt   = logSt[mask]
        n_s  = mask.sum()

        if n_s < 10:
            results[name] = dict(theta=0.3, mu=np.log(80/10000), sigma=0.15,
                                 lam=1.0, jmu=[-0.02, 0.02], jsig=[0.02, 0.05],
                                 jw=[0.5, 0.5])
            print(f"  [{label}|{name}] n={n_s} too few -- using defaults")
            continue

        sample_mean = np.mean(Yt)
        if len(Yt) > 1:
            rho = np.corrcoef(Yt[:-1], Yt[1:])[0, 1]
            rho = np.clip(rho, 0.001, 0.999)
        else:
            rho = 0.5

        # Estimate then constrain to sensible bounds
        theta = np.clip(-np.log(rho) / dt_obs, 0.1, 5.0)
        mu_ou = sample_mean
        sigma = np.clip(np.std(dY) / np.sqrt(dt_obs), 0.05, 0.50)

        # Jump detection: residuals > 2.5 sigma_resid
        drift_fit = theta * (mu_ou - Yt) * dt_obs
        resid     = dY - drift_fit
        sig_r     = np.std(resid)
        jmask     = np.abs(resid) > 2.5 * sig_r
        lam       = jmask.sum() / max(n_s * dt_obs, 0.01)

        # Fit 2-component Gaussian mixture to jump residuals
        jres = resid[jmask]
        if len(jres) >= 4:
            pos = jres[jres > 0]
            neg = jres[jres <= 0]
            jmu  = [np.mean(neg) if len(neg) > 0 else -0.02,
                    np.mean(pos) if len(pos) > 0 else  0.02]
            jsig = [max(np.std(neg), 0.005) if len(neg) > 1 else 0.02,
                    max(np.std(pos), 0.005) if len(pos) > 1 else 0.02]
            jw   = [len(neg)/len(jres), len(pos)/len(jres)]
            jmu  = [max(jmu[0], -0.30), min(jmu[1], 0.30)]
            jsig = [min(s, 0.20) for s in jsig]
        else:
            jmu  = [0.0, 0.02]
            jsig = [0.01, 0.02]
            jw   = [0.5, 0.5]

        results[name] = dict(theta=theta, mu=mu_ou, sigma=sigma,
                             lam=lam, jmu=jmu, jsig=jsig, jw=jw)
        if label:
            print(f"  [{label}|{name}] theta={theta:.3f}  mu_exp={np.exp(mu_ou)*10000:.0f}bps  "
               f"sigma={sigma:.3f}  lam={lam:.2f}  n={n_s}")

    return results

def weighted_params(p_r, p_o, wr=0.70, wo=0.30):
    combined = {}
    for state in ["normal", "crisis"]:
        pr, po = p_r[state], p_o[state]
        combined[state] = dict(
            theta = wr*pr["theta"] + wo*po["theta"],
            mu    = wr*pr["mu"]    + wo*po["mu"],
            sigma = wr*pr["sigma"] + wo*po["sigma"],
            lam   = wr*pr["lam"]   + wo*po["lam"],
            jmu   = [wr*pr["jmu"][i] + wo*po["jmu"][i]  for i in range(2)],
            jsig  = [wr*pr["jsig"][i]+ wo*po["jsig"][i] for i in range(2)],
            jw    = [wr*pr["jw"][i]  + wo*po["jw"][i]   for i in range(2)],
        )
    return combined

# Calibrate both windows at monthly frequency
print("=== RECENT window 2020-2026 (monthly) ===")
reg_recent, _, _ = detect_regimes(df_recent_m)
cf_recent_m = reg_recent["is_crisis"].values
par_recent  = fit_log_ou_regime(df_recent_m["cdx_unified_bps"],
                                 cf_recent_m, dt_obs=1/12, label="recent-monthly")

print("\n=== OLD window 2007-2016 (monthly) ===")
reg_old, _, _ = detect_regimes(df_old_m)
cf_old = reg_old["is_crisis"].values
par_old = fit_log_ou_regime(df_old_m["cdx_unified_bps"],
                             cf_old, dt_obs=1/12, label="old-monthly")

# Weighted composite 70/30
CALIB = weighted_params(par_recent, par_old, W_RECENT, W_OLD)

print("\n=== COMPOSITE (70/30 weighted, both monthly) ===")
print(f"\nS0 kept at market level: {S0_BPS:.1f} bps  (set in Cell 03)")
for state in ["normal", "crisis"]:
    p = CALIB[state]
    print(f"  [{state}] theta={p['theta']:.3f}  mu_exp={np.exp(p['mu'])*10000:.0f}bps  "
          f"sigma={p['sigma']:.3f}  lam={p['lam']:.2f}/yr")


In [ ]:
# CELL 07b -- OU Parameter Bootstrap Confidence Intervals
# Quantifies statistical uncertainty in theta, mu, sigma estimates

N_BOOT = 500

def bootstrap_ou_params(spread_series, crisis_flags, dt_obs, n_boot=500, seed=42):
    np.random.seed(seed)
    S      = spread_series.dropna().values
    n      = len(S)
    results = {"normal": {"theta":[], "mu":[], "sigma":[], "lam":[]},
               "crisis": {"theta":[], "mu":[], "sigma":[], "lam":[]}}

    for _ in range(n_boot):
        idx    = np.random.choice(n, size=n, replace=True)
        idx    = np.sort(idx)
        S_boot = S[idx]
        cf_boot = crisis_flags[-len(S_boot)+1:] if len(crisis_flags) >= len(S_boot) else crisis_flags

        import io, contextlib
        with contextlib.redirect_stdout(io.StringIO()):
            par = fit_log_ou_regime(
                pd.Series(S_boot), cf_boot, dt_obs=dt_obs, label="")

        for state in ["normal", "crisis"]:
            results[state]["theta"].append(par[state]["theta"])
            results[state]["mu"].append(np.exp(par[state]["mu"]) * 10_000)
            results[state]["sigma"].append(par[state]["sigma"])
            results[state]["lam"].append(par[state]["lam"])

    return results

print(f"Bootstrapping OU parameters ({N_BOOT} samples)...")
boot_rec = bootstrap_ou_params(
    df_recent_m["cdx_unified_bps"], crisis_flags, dt_obs=1/12, n_boot=N_BOOT)
boot_old = bootstrap_ou_params(
    df_old_m["cdx_unified_bps"],    crisis_flags, dt_obs=1/12, n_boot=N_BOOT)

print("\nOU Parameter Bootstrap 95% Confidence Intervals")
print("=" * 70)
for window, boot in [("Recent (2020-2026)", boot_rec),
                     ("Old (2007-2016)",    boot_old)]:
    print(f"\n  {window}:")
    print(f"  {'Param':>8}  {'Regime':>8}  {'Estimate':>10}  "
          f"{'95% CI Lower':>13}  {'95% CI Upper':>13}")
    print("  " + "-" * 58)
    for state in ["normal", "crisis"]:
        for param in ["theta", "mu", "sigma", "lam"]:
            vals = boot[state][param]
            lo   = np.percentile(vals, 2.5)
            hi   = np.percentile(vals, 97.5)
            est  = np.mean(vals)
            print(f"  {param:>8}  {state:>8}  {est:>10.4f}  "
                  f"{lo:>13.4f}  {hi:>13.4f}")
print("=" * 70)

# ── Chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("OU Parameter Bootstrap Distributions (500 samples)",
             fontsize=12, fontweight="bold")

params     = ["theta", "mu", "sigma", "lam"]
param_lbls = ["θ (mean reversion)", "μ_exp (bps)", "σ (volatility)", "λ (jump rate)"]

for row, (window, boot) in enumerate([("Recent (2020-2026)", boot_rec),
                                       ("Old (2007-2016)",    boot_old)]):
    for col, (param, lbl) in enumerate(zip(params, param_lbls)):
        ax = axes[row, col]
        for state, color in [("normal", "#1976D2"), ("crisis", "#E53935")]:
            vals = boot[state][param]
            ax.hist(vals, bins=30, alpha=0.5, color=color,
                    density=True, label=state)
            lo = np.percentile(vals, 2.5)
            hi = np.percentile(vals, 97.5)
            ax.axvline(lo, color=color, ls="--", lw=1)
            ax.axvline(hi, color=color, ls="--", lw=1)
        ax.set_title(f"{lbl}\n{window}", fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("../figures/experimental_bootstrap_ci.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 07c -- Blending Weight Sensitivity (70/30 vs alternatives)
# Shows how OU parameters change across different recent/old weightings
# No MC run needed -- just parameter sensitivity

import io, contextlib

WEIGHT_COMBOS = [
    (0.50, 0.50),
    (0.60, 0.40),
    (0.70, 0.30),  # base
    (0.80, 0.20),
    (0.90, 0.10),
]

print("Blending Weight Sensitivity — Effect on Calibrated OU Parameters")
print("=" * 85)
print(f"  {'W_Rec':>6}  {'W_Old':>6}  {'Normal θ':>10}  {'Normal μ':>10}  "
      f"{'Normal σ':>10}  {'Crisis θ':>10}  {'Crisis μ':>10}  {'Crisis σ':>10}")
print("  " + "-" * 78)

for w_rec, w_old in WEIGHT_COMBOS:
    with contextlib.redirect_stdout(io.StringIO()):
        par_rec = fit_log_ou_regime(
            df_recent_m["cdx_unified_bps"], cf_recent_m, dt_obs=1/12, label="")
        par_old = fit_log_ou_regime(
            df_old_m["cdx_unified_bps"], cf_old, dt_obs=1/12, label="")
        calib_w = weighted_params(par_rec, par_old, w_rec, w_old)

    marker = "  ◄ base" if w_rec == 0.70 else ""
    print(f"  {w_rec:>6.0%}  {w_old:>6.0%}  "
          f"{calib_w['normal']['theta']:>10.3f}  "
          f"{np.exp(calib_w['normal']['mu'])*10000:>10.1f}  "
          f"{calib_w['normal']['sigma']:>10.3f}  "
          f"{calib_w['crisis']['theta']:>10.3f}  "
          f"{np.exp(calib_w['crisis']['mu'])*10000:>10.1f}  "
          f"{calib_w['crisis']['sigma']:>10.3f}{marker}")  # ← added Crisis σ

print("=" * 85)

## CELL 08 — Spread Simulator


In [ ]:
# CELL 08 -- Regime-Switching Log-OU Spread Simulator (monthly step)
# Three components per month:
#   Drift  = mean reversion toward long-run average (theta, mu)
#   Noise  = random volatility using fat-tailed Student-t (sigma)
#   Jump   = occasional sudden spike/drop (lam, jmu, jsig, jw)

def spread_step_monthly(logS, regime, calib, dt=DT):
    p     = calib[regime]
    drift = p["theta"] * (p["mu"] - logS) * dt
    noise = p["sigma"] * np.sqrt(dt) * t_dist.rvs(df=5)   
    jump  = 0.0
    if np.random.rand() < p["lam"] * dt:
        comp = np.random.choice([0,1], p=p["jw"])
        jump = np.random.normal(p["jmu"][comp], p["jsig"][comp])
    logS_new = logS + drift + noise + jump
    # Floor: 1 bps, Ceiling: 400 bps (CDX IG historical max ~280 bps in 2008)
    return np.clip(logS_new, np.log(1e-4), np.log(0.04))

def next_regime(reg, P):
    return np.random.choice([0,1], p=P[reg])

def regime_name(r):
    return "crisis" if r == 1 else "normal"

# Sanity check: generate one sample path
S0_dec = S0_BPS / 10_000
logS0  = np.log(S0_dec)
path_bps = [S0_BPS]
reg = 0
for _ in range(N_PERIODS):
    reg   = next_regime(reg, P_markov)
    logS0 = spread_step_monthly(logS0, regime_name(reg), CALIB)
    path_bps.append(np.exp(logS0)*10_000)
print("Sample path (bps):", np.round(path_bps[::6], 1))  # semi-annual points


## CELL 10 — Duration,Leverage


In [ ]:
# CELL 10 -- Duration and Shortfall Leverage

MIN_LEV = 3

def cdx_duration(S_bps, rf_pct, maturity_yrs, freq=2, lgd=LGD):
    """Risky annuity — unchanged. LGD=60% stays in discount rate."""
    S_dec    = max(S_bps, 1) / 10_000
    rf_dec   = max(rf_pct, 0) / 100
    discount = rf_dec + S_dec / lgd
    dt_s     = 1 / freq
    n_steps  = max(int(maturity_yrs * freq), 1)
    return sum(dt_s * np.exp(-discount * dt_s * j)
               for j in range(1, n_steps + 1))
def dynamic_surplus_level(t, face, cpn_semi):
    periods_left = N_PERIODS - t
    n_coupons    = periods_left // 6
    return face + cpn_semi * n_coupons
def shortfall_notional(t, nav, s_bps, cpn_semi, frozen=False, current=None):

    if frozen and current is not None:
        return current                        # crisis: do nothing

    mat_remaining = (N_PERIODS - t) / 12
    if mat_remaining < 1/12:                   # near-maturity edge case
        return MIN_LEV * nav

    s_dec     = max(s_bps, 1) / 10_000
    dur       = cdx_duration(s_bps, RF_PCT, mat_remaining)
    v_star    = dynamic_surplus_level(t, FACE_VALUE, cpn_semi)
    shortfall = max(v_star - nav, 0)          # floor: never short index

    denom = s_dec * dur
    if denom < 1e-8:
        return MIN_LEV * nav                # denominator collapse edge case

    raw_lv = (shortfall / denom) / nav if nav > 1 else 0
    raw_lv = min(raw_lv, LEV_CAP)            # single 15x hard cap
    return raw_lv  * nav # leverage * current notional

print("shortfall_notional() ready — freeze in crisis, formula in normal.")
print("Duration verification:")
for s, r, mat in [(50,4.0,10),(80,4.5,10),(150,5.0,10),(280,5.5,5)]:
    print(f"  S={s:>3}bps rf={r}% mat={mat}yr → dur={cdx_duration(s,r,mat):.3f}")

## CELL 11 — Monthly NAV Update (P&L Engine)


In [ ]:
# CELL 11 -- Monthly NAV Update
# Returns PRE-coupon NAV so Cell 12 can detect coupon shortfall events
# Income:  carry (spread × notional) + MTM + cash interest
# Costs:   defaults (Poisson) + admin fee + facility fee

def nav_update_monthly(nav, cash_res, logS_old, logS_new, notional,
                       r_f_monthly, periods_left_months, lgd=LGD):
    """Returns (nav_pre_coupon, cash_new, n_def).
    Coupon deduction moved to caller so coupon-shortfall events are detectable."""
    S_new_dec = np.exp(logS_new)
    S_old_dec = np.exp(logS_old)
    mat_yrs   = max(periods_left_months / 12, 1/12)
    dur       = cdx_duration(S_new_dec*10_000, r_f_monthly*12*100, mat_yrs)

    # ── Income ────────────────────────────────────────────────────────────────
    running_income = S_old_dec * notional * DT   # spread carry on leveraged position
    net_carry      = running_income
    mtm            = (S_old_dec - S_new_dec) * dur * notional  # +ve if spreads tightened
    cash_int       = r_f_monthly * cash_res * DT               # Treasury rate on cash reserve

    # ── Fees ──────────────────────────────────────────────────────────────────
    admin_fee    = ADMIN_MONTH                                  # fixed monthly, on face value
    facility_fee = (FACILITY_FEE_BPS / 10_000) * notional * DT # variable, on leveraged notional

    fees = admin_fee + facility_fee

    nav_pre_cpn = nav + net_carry + mtm + cash_int - fees
    cash_new    = max(nav_pre_cpn - notional*0.02, 0)
    return nav_pre_cpn, cash_new


## CELL 12 — Monte Carlo Simulation Loop


In [ ]:
def run_mc(coupon_bps, n_paths=N_PATHS, lgd=LGD,
           label="", verbose=True, seed=None,
           s0_override=None):
    if seed is not None:
        np.random.seed(seed)

    S0_use = s0_override if s0_override is not None else S0_BPS
    logS0  = np.log(S0_use / 10_000)
    r_f_monthly = (RF_PCT / 100) / 12
    cpn_semi    = (coupon_bps / 10_000) * FACE_VALUE * 0.5

    nav_paths    = np.zeros((n_paths, N_PERIODS + 1))
    spread_paths = np.zeros((n_paths, N_PERIODS + 1))
    lev_paths    = np.zeros((n_paths, N_PERIODS))
    co_flag      = np.zeros(n_paths, dtype=bool)
    cs_flag      = np.zeros(n_paths, dtype=bool)
    co_period    = np.full(n_paths, -1, dtype=int)
    final_navs   = np.zeros(n_paths)
    default_type = np.full(n_paths, "survived", dtype="<U22")

    for p in range(n_paths):
        nav            = NAV0
        cash_res       = 0.0
        logS           = logS0
        reg            = 0
        notional       = FACE_VALUE
        surplus_locked = False

        nav_paths[p, 0]    = nav
        spread_paths[p, 0] = np.exp(logS) * 10_000

        for t in range(1, N_PERIODS + 1):
            reg     = next_regime(reg, P_markov)
            reg_str = regime_name(reg)
            in_crisis = (reg == 1)

            # ── LEVERAGE ──────────────────────────────────────────
            # Crisis  → freeze: hold notional exactly as-is
            # Normal  → shortfall formula, 15x cap, 3x floor
            notional = shortfall_notional(
                t, nav, np.exp(logS) * 10_000, cpn_semi,
                frozen=in_crisis,
                current=notional
            )
            lev_paths[p, t-1] = notional / max(nav, 1)

            # ── SPREAD STEP ───────────────────────────────────────
            logS_new           = spread_step_monthly(logS, reg_str, CALIB)
            spread_paths[p, t] = np.exp(logS_new) * 10_000

            is_cpn       = (t % 6 == 0)
            periods_left = N_PERIODS - t

            # ── NAV UPDATE ────────────────────────────────────────
            nav_pre_cpn, cash_res = nav_update_monthly(
                nav, cash_res, logS, logS_new,
                notional, r_f_monthly, periods_left)

            # ── EVENT 1: COUPON SHORTFALL ─────────────────────────
            if is_cpn and nav_pre_cpn < cpn_semi:
                cs_flag[p]      = True
                default_type[p] = "coupon_shortfall"
                co_period[p]    = t
                nav_paths[p, t:]      = max(nav_pre_cpn, 0)
                spread_paths[p, t+1:] = np.exp(logS_new) * 10_000
                break

            nav = (nav_pre_cpn - cpn_semi) if is_cpn else nav_pre_cpn

            # ── EVENT 2: SURPLUS LOCK ─────────────────────────────
            if not surplus_locked:
                v_star = dynamic_surplus_level(t, FACE_VALUE, cpn_semi)
                if nav >= v_star:
                    notional       = 1 * nav
                    surplus_locked = True

            nav_paths[p, t] = nav
            logS = logS_new

            # ── EVENT 3: CASH-OUT ─────────────────────────────────
            if nav <= CASHOUT_LEVEL:
                co_flag[p]      = True
                default_type[p] = "cashout"
                co_period[p]    = t
                nav_paths[p, t+1:]    = max(nav, 0)
                spread_paths[p, t+1:] = np.exp(logS_new) * 10_000
                break

        final_navs[p] = nav

    # ── EVENT 4: MATURITY SHORTFALL ───────────────────────────────
    survived        = ~(cs_flag | co_flag)
    mat_short_flag  = survived & (final_navs < FACE_VALUE)
    any_default     = cs_flag | co_flag | mat_short_flag

    default_type = np.where(mat_short_flag, "principal_shortfall", default_type)

    pcs  = cs_flag.mean()
    pco  = co_flag.mean()
    pmat = mat_short_flag.mean()
    pcmb = any_default.mean()

    if verbose:
        print(f"  Coupon {coupon_bps}bps | {n_paths:,} paths")
        print(f"  P(coupon shortfall) = {pcs:.4%}")
        print(f"  P(cash-out)         = {pco:.4%}")
        print(f"  P(maturity short)   = {pmat:.4%}")
        print(f"  P(any default)      = {pcmb:.4%}  <-- headline")
        print(f"  E[NAV at maturity]  = ${final_navs.mean():,.0f}")
        print(f"  Median NAV          = ${np.median(final_navs):,.0f}")

    return dict(
        nav_paths=nav_paths, spread_paths=spread_paths,
        lev_paths=lev_paths, co_flag=co_flag, cs_flag=cs_flag,
        mat_short_flag=mat_short_flag, any_default=any_default,
        co_period=co_period, default_type=default_type,
        final_navs=final_navs,
        pco=pco, pcs=pcs, pmat=pmat, pcmb=pcmb,
        enav=final_navs.mean(), median_nav=np.median(final_navs)
    )

print("run_mc() ready — shortfall leverage, crisis freeze.")

Test:Mean Leverage Per Year

In [ ]:
np.random.seed(42)
r = run_mc(50, n_paths=500, verbose=False)
lev = r["lev_paths"]
print("Mean leverage by year:")
for yr in [1, 2, 3, 5, 7, 10]:
    t = yr * 12
    print(f"  Year {yr:>2}: {lev[:, t-1].mean():.2f}x")

## CELL 13 — Coupon Solver (Common Random Numbers)


In [ ]:
# CELL 13 -- Coupon Solver with Common Random Numbers (CRN)
# ─────────────────────────────────────────────────────────────────
# Tests every coupon from 10 to 50 bps in steps of 10
# CRN: same random seed for all coupons -- differences due to
#      coupon size only, not randomness
# Pass criteria:
#   P(any default) < 18%  (Moody's Ba 10yr cumulative PD threshold)
#   Median NAV >= par     (median more robust than mean to outliers)
# Selects highest passing coupon as optimal
# ─────────────────────────────────────────────────────────────────

SWEEP_SEED        = 42       # locked: every coupon sees identical paths
SWEEP_PATHS       = 2_000    # CRN controls variance -- 2k sufficient
BA_10YR_THRESHOLD = 0.18     # Moody's Ba 10yr cumulative PD threshold

print(f"Sweeping coupons 10 to 50 bps ({SWEEP_PATHS:,} paths, "
      f"CRN seed={SWEEP_SEED})...")
print(f"Pass: P(any default) < {BA_10YR_THRESHOLD:.0%}  "
      f"AND  Median NAV >= par")
print()
print(f"{'Cpn':>4}  {'P(cs)':>7}  {'P(CO)':>7}  {'P(mat)':>7}  "
      f"{'P(any)':>7}  {'E[NAV]':>12}  {'Med[NAV]':>12}  Pass?")
print("-" * 80)

sweep = []
for cpn in range(10, 55, 10):
    r  = run_mc(cpn, n_paths=SWEEP_PATHS, verbose=False, seed=SWEEP_SEED)
    ok = (r["pcmb"] < BA_10YR_THRESHOLD) and (r["median_nav"] >= FACE_VALUE)
    sweep.append({"cpn"        : cpn,
                  "pcs"        : r["pcs"],
                  "pco"        : r["pco"],
                  "pmat"       : r["pmat"],
                  "pcmb"       : r["pcmb"],
                  "enav"       : r["enav"],
                  "median_nav" : r["median_nav"],
                  "ok"         : ok})
    print(f"{cpn:>4}  {r['pcs']:>7.2%}  {r['pco']:>7.2%}  {r['pmat']:>7.2%}  "
          f"{r['pcmb']:>7.2%}  ${r['enav']:>10,.0f}  "
          f"${r['median_nav']:>10,.0f}  {'YES' if ok else 'NO'}")

df_sw       = pd.DataFrame(sweep)
feas        = df_sw[df_sw["ok"]]["cpn"]

if feas.empty:
    print("\nWARNING: No feasible coupon found -- structure not viable")
    print("         Check calibration parameters and leverage settings")
    OPTIMAL_CPN = None
else:
    OPTIMAL_CPN = int(feas.max())
    print(f"\n  Optimal coupon = {OPTIMAL_CPN} bps")

## CELL 14 — Full Monte Carlo at Optimal Coupon


In [ ]:
# CELL 14 -- Full Monte Carlo at Optimal Coupon (10,000 paths)
print(f"Running {N_PATHS:,} paths at {OPTIMAL_CPN} bps (monthly timestep, 10yr)...")
mc = run_mc(OPTIMAL_CPN, n_paths=N_PATHS, verbose=True)
print("Done.")


## CELL 15 — Probability of Default & Rating Report


In [ ]:
# CELL 15 -- PD Estimation & Rating Assignment 
# ─────────────────────────────────────────────────────────────────

co_flag    = mc["co_flag"]
cs_flag    = mc["cs_flag"]       
co_period  = mc["co_period"]
final_navs = mc["final_navs"]

# ── Clean, non-overlapping event flags ───────────────────────────
cashout_default = co_flag
coupon_default  = cs_flag & ~co_flag          # coupon short but not also a cash-out
survivors       = ~co_flag & ~cs_flag         # paths that reached maturity cleanly
mat_default     = survivors & (final_navs < FACE_VALUE)  # shortfall at maturity only

combined_default = cashout_default | coupon_default | mat_default

# ── PD decomposition ─────────────────────────────────────────────
pd_cashout  = cashout_default.mean()
pd_coupon   = coupon_default.mean()
pd_maturity = mat_default.mean()
pd_combined = combined_default.mean()

# ── Wilson 95% CI on combined PD ─────────────────────────────────
z      = 1.96
denom  = 1 + z**2 / N_PATHS
centre = (pd_combined + z**2 / (2 * N_PATHS)) / denom
margin = z * np.sqrt(pd_combined * (1 - pd_combined) / N_PATHS
                     + z**2 / (4 * N_PATHS**2)) / denom
pd_lo, pd_hi = max(centre - margin, 0), centre + margin

# ── Expected loss ─────────────────────────────────────────────────
# Recovery is only meaningful when defaults actually occurred
defaulted_navs = final_navs[combined_default]
if len(defaulted_navs) > 0:
    rec_given_def = defaulted_navs.mean() / FACE_VALUE
    expected_loss = pd_combined * (1 - rec_given_def)
else:
    rec_given_def = None   # undefined — no defaults observed
    expected_loss = 0.0

# ── Illustrative thresholds only; not an external credit rating ──
RATING_SCALE = {
    "Aaa"  : 0.0009,
    "Aa"   : 0.0032,
    "A"    : 0.0067,
    "Baa"  : 0.0141,
    "Ba"   : 0.0825,
    "B"    : 0.2063,
    "Caa-C": 0.3410,
}
SP_EQ = {"Aaa":"AAA","Aa":"AA","A":"A","Baa":"BBB",
          "Ba":"BB","B":"B","Caa-C":"CCC"}

def assign_rating(pd_val):
    for r, thresh in RATING_SCALE.items():
        if pd_val <= thresh: return r
    return "Below Caa-C"

assigned = assign_rating(pd_combined)
r_lo     = assign_rating(pd_lo)
r_hi     = assign_rating(pd_hi)

print("=" * 55)
print("  CPDO — MODEL EVENT SUMMARY (10-YEAR; NOT A CREDIT RATING)")
print("=" * 55)
print(f"  Paths              : {N_PATHS:,}")
print(f"  Optimal coupon     : {OPTIMAL_CPN} bps")
print()
print(f"  P(coupon shortfall): {pd_coupon:.4%}")
print(f"  P(cash-out)        : {pd_cashout:.4%}")
print(f"  P(mat shortfall)   : {pd_maturity:.4%}")
print(f"  P(combined)        : {pd_combined:.4%}  <-- headline")
print(f"  95% CI             : [{pd_lo:.4%},  {pd_hi:.4%}]")
if rec_given_def is not None:
    print(f"  Recovery given def : {rec_given_def:.4%}")
    print(f"  Expected loss      : {expected_loss:.4%}")
else:
    print(f"  Recovery given def : n/a (no defaults observed)")
    print(f"  Expected loss      : 0.0000%")
print()
print(f"  Assigned rating    : {assigned} / {SP_EQ.get(assigned,'--')}")
print(f"  95% CI range       : {r_lo} to {r_hi}")
print("=" * 55)

## CELL 16 — NAV Fan Chart & Leverage Path


In [ ]:
# CELL 16
months = np.arange(N_PERIODS+1)
lev_mo = np.arange(1, N_PERIODS+1)
nav_m  = mc["nav_paths"]
lev_m  = mc["lev_paths"]

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle(
    f"CPDO Monte Carlo -- {N_PATHS:,} Paths | Coupon {OPTIMAL_CPN}bps | "
    f"Regime Log-OU + Dorn | 10yr Monthly",
    fontsize=11, fontweight="bold")

ax = axes[0]
p5,p25,p50,p75,p95 = np.percentile(nav_m,[5,25,50,75,95],axis=0)
ax.fill_between(months,p5/1e6, p95/1e6,alpha=0.15,color="#1976D2",label="5th-95th pct")
ax.fill_between(months,p25/1e6,p75/1e6,alpha=0.35,color="#1976D2",label="25th-75th pct")
ax.plot(months,nav_m.mean(axis=0)/1e6,color="#1976D2",lw=2.5,label="Mean")
ax.plot(months,p50/1e6,color="black",lw=1.5,ls="--",label="Median")
ax.axhline(1.0,color="black",ls=":",lw=1.5,label="Par ($1M)")
ax.axhline(SAFETY_BUFFER,color="red",ls=":",lw=1.2,label=f"Cash-out ({SAFETY_BUFFER:.0%})")
ax.set_xlabel("Months"); ax.set_ylabel("NAV ($M)")
ax.set_title("NAV Fan Chart (10yr)"); ax.legend(fontsize=8)

ax = axes[1]
lp5,lp95 = np.percentile(lev_m,[5,95],axis=0)
ax.fill_between(lev_mo,lp5,lp95,alpha=0.20,color="#43A047",label="5th-95th pct")
ax.plot(lev_mo,lev_m.mean(axis=0),color="#43A047",lw=2.5,label="Mean leverage")
ax.set_xlabel("Months"); ax.set_ylabel("Leverage (x face)")
ax.axhline(LEV_CAP, color="red", ls="--", lw=2,
           label=f"Hard cap ({LEV_CAP}x)")

plt.tight_layout()
plt.savefig("../figures/experimental_nav_leverage.png",dpi=150,bbox_inches="tight")
plt.show()


## CELL 17 — Simulated Spread Paths


In [ ]:
# CELL 17

N_SHOW = 20
years  = np.arange(N_PERIODS + 1) / 12

def run_spread_paths(n=500, with_jumps=True, seed=42):
    np.random.seed(seed)
    logS0 = np.log(S0_BPS / 10_000)
    paths = np.zeros((n, N_PERIODS + 1))
    paths[:, 0] = S0_BPS
    for p in range(n):
        logS = logS0
        reg  = 0
        for t in range(1, N_PERIODS + 1):
            reg     = next_regime(reg, P_markov)
            reg_str = regime_name(reg)
            par     = CALIB[reg_str]
            drift   = par["theta"] * (par["mu"] - logS) * DT
            noise   = par["sigma"] * np.sqrt(DT) * t_dist.rvs(df=5)
            jump    = 0.0
            if with_jumps and np.random.rand() < par["lam"] * DT:
                comp = np.random.choice([0, 1], p=par["jw"])
                jump = np.random.normal(par["jmu"][comp], par["jsig"][comp])
            logS = np.clip(logS + drift + noise + jump,
                           np.log(1e-4), np.log(0.04))
            paths[p, t] = np.exp(logS) * 10_000
    return paths

# then the rest of your Cell 18 code below...
paths_base = run_spread_paths(n=500, with_jumps=False)
paths_jump = run_spread_paths(n=500, with_jumps=True)

ou_mean_bps = np.exp(CALIB["normal"]["mu"]) * 10_000

fig, ax = plt.subplots(figsize=(14, 5))

for i in range(N_SHOW):
    ax.plot(years, paths_base[i], color="#1976D2", alpha=0.25, lw=0.8)
for i in range(N_SHOW):
    ax.plot(years, paths_jump[i], color="#FF9800", alpha=0.25, lw=0.8)

ax.plot(years, paths_base.mean(axis=0), color="#1976D2", lw=2.5,
        label="Log-OU (no jumps)")
ax.plot(years, paths_jump.mean(axis=0), color="#FF9800", lw=2.5,
        label="Log-OU + Jumps")

ax.axhline(ou_mean_bps, color="red", ls="--", lw=1.5,
           label=f"OU long-run mean = {ou_mean_bps:.0f} bps")
ax.axhline(S0_BPS, color="black", ls=":", lw=1.2,
           label=f"S0 = {S0_BPS:.0f} bps")

ax.set_xlabel("Years")
ax.set_ylabel("Spread (bps)")
ax.set_title(f"{N_SHOW} Simulated Paths: Log-OU vs Log-OU + Jumps (10yr)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0)
ax.grid(True, alpha=0.15)
fig.tight_layout()
plt.show()


# ── Chart 2: Spread Distribution ─────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

hist_vals = aligned_m["cdx_unified_bps"].dropna()

ax.hist(hist_vals, bins=30, density=True, color="#43A047",
        alpha=0.6, label="Historical")
ax.hist(paths_base.flatten(), bins=80, density=True, color="#1976D2",
        alpha=0.5, label="Log-OU (no jumps)")
ax.hist(paths_jump.flatten(), bins=80, density=True, color="#FF9800",
        alpha=0.5, label="Log-OU + Jumps")

ax.axvline(ou_mean_bps, color="red", ls="--", lw=1.5,
           label=f"OU mean = {ou_mean_bps:.0f} bps")
ax.axvline(S0_BPS, color="black", ls=":", lw=1.5,
           label=f"S0 = {S0_BPS:.0f} bps")

ax.set_xlabel("Spread (bps)")
ax.set_ylabel("Density")
ax.set_title("Spread Distribution: Historical vs Log-OU vs Log-OU + Jumps",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)
fig.tight_layout()
plt.show()

In [ ]:
# CELL 17b -- Distribution Fit Validation
# QQ plots + KS test: simulated vs historical
# Formally tests if jump model improves fit over no-jump model

from scipy import stats

hist_vals   = aligned_m["cdx_unified_bps"].dropna().values
sim_base    = paths_base.flatten()
sim_jump    = paths_jump.flatten()

# Subsample to match historical size for fair comparison
np.random.seed(42)
sim_base_s  = np.random.choice(sim_base, size=len(hist_vals), replace=False)
sim_jump_s  = np.random.choice(sim_jump, size=len(hist_vals), replace=False)

# ── KS Tests ─────────────────────────────────────────────────────
ks_base = stats.ks_2samp(hist_vals, sim_base_s)
ks_jump = stats.ks_2samp(hist_vals, sim_jump_s)

print("=" * 55)
print("  Distribution Fit: KS Test (Simulated vs Historical)")
print("=" * 55)
print(f"  Historical n     : {len(hist_vals)}")
print(f"  Simulated n      : {len(sim_base_s)}")
print()
print(f"  Log-OU (no jumps):")
print(f"    KS statistic   : {ks_base.statistic:.4f}")
print(f"    p-value        : {ks_base.pvalue:.4f}")
print(f"    Conclusion     : {'FAIL to reject H0 (good fit)' if ks_base.pvalue > 0.05 else 'REJECT H0 (poor fit)'}")
print()
print(f"  Log-OU + Jumps:")
print(f"    KS statistic   : {ks_jump.statistic:.4f}")
print(f"    p-value        : {ks_jump.pvalue:.4f}")
print(f"    Conclusion     : {'FAIL to reject H0 (good fit)' if ks_jump.pvalue > 0.05 else 'REJECT H0 (poor fit)'}")
print()
print(f"  Jump improvement : KS stat reduced by "
      f"{(ks_base.statistic - ks_jump.statistic)/ks_base.statistic*100:.1f}%")
print("=" * 55)

# ── Charts ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Distribution Fit Validation — Simulated vs Historical Spreads",
             fontsize=12, fontweight="bold")

# ── Left: Density overlay ─────────────────────────────────────────
ax = axes[0]
bins = np.linspace(0, max(hist_vals.max(), sim_jump.max()), 60)
ax.hist(hist_vals,  bins=bins, density=True, color="#43A047",
        alpha=0.5, label="Historical")
ax.hist(sim_base_s, bins=bins, density=True, color="#1976D2",
        alpha=0.5, label="Log-OU (no jumps)")
ax.hist(sim_jump_s, bins=bins, density=True, color="#FF9800",
        alpha=0.5, label="Log-OU + Jumps")
ax.set_xlabel("Spread (bps)")
ax.set_ylabel("Density")
ax.set_title("Density Overlay")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Middle: QQ plot — no jumps ────────────────────────────────────
ax = axes[1]
hist_sorted    = np.sort(hist_vals)
sim_base_sorted = np.sort(np.random.choice(sim_base,
                          size=len(hist_vals), replace=False))
ax.scatter(hist_sorted, sim_base_sorted, alpha=0.4,
           s=8, color="#1976D2", label="Log-OU vs Historical")
mn = min(hist_sorted.min(), sim_base_sorted.min())
mx = max(hist_sorted.max(), sim_base_sorted.max())
ax.plot([mn, mx], [mn, mx], color="red", lw=1.5, ls="--",
        label="Perfect fit line")
ax.set_xlabel("Historical quantiles (bps)")
ax.set_ylabel("Simulated quantiles (bps)")
ax.set_title(f"QQ Plot — Log-OU (no jumps)\nKS={ks_base.statistic:.4f}, "
             f"p={ks_base.pvalue:.3f}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Right: QQ plot — with jumps ───────────────────────────────────
ax = axes[2]
sim_jump_sorted = np.sort(np.random.choice(sim_jump,
                          size=len(hist_vals), replace=False))
ax.scatter(hist_sorted, sim_jump_sorted, alpha=0.4,
           s=8, color="#FF9800", label="Log-OU+Jumps vs Historical")
ax.plot([mn, mx], [mn, mx], color="red", lw=1.5, ls="--",
        label="Perfect fit line")
ax.set_xlabel("Historical quantiles (bps)")
ax.set_ylabel("Simulated quantiles (bps)")
ax.set_title(f"QQ Plot — Log-OU + Jumps\nKS={ks_jump.statistic:.4f}, "
             f"p={ks_jump.pvalue:.3f}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("../figures/experimental_distribution_fit.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Tail fit comparison ───────────────────────────────────────────
print("\nTail Fit Comparison (95th percentile and above):")
hist_95  = np.percentile(hist_vals,  95)
base_95  = np.percentile(sim_base_s, 95)
jump_95  = np.percentile(sim_jump_s, 95)
hist_99  = np.percentile(hist_vals,  99)
base_99  = np.percentile(sim_base_s, 99)
jump_99  = np.percentile(sim_jump_s, 99)

print(f"  {'':20}  {'Historical':>12}  {'Log-OU':>10}  {'Log-OU+Jump':>12}")
print(f"  {'95th percentile':20}  {hist_95:>12.1f}  {base_95:>10.1f}  {jump_95:>12.1f}")
print(f"  {'99th percentile':20}  {hist_99:>12.1f}  {base_99:>10.1f}  {jump_99:>12.1f}")
print(f"  {'Max':20}  {hist_vals.max():>12.1f}  "
      f"{sim_base_s.max():>10.1f}  {sim_jump_s.max():>12.1f}")

In [ ]:
#CELL 17c -- Autocorrelation Validation
# Tests if Log-OU autocorrelation structure matches historical data
import pandas as pd
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Log-OU Autocorrelation Validation",
             fontsize=12, fontweight="bold")

# ── Historical log-spread returns ─────────────────────────────────
hist_dlogS = regime_df["dlog"].dropna().values
hist_logS = regime_df.loc[regime_df["dlog"].notna(), "logS"].values

# ── Simulated log-spread returns ──────────────────────────────────
sim_logS_paths = np.log(paths_base[:50] / 10_000)
sim_dlogS = np.diff(sim_logS_paths, axis=1).ravel()

# ── Left: ACF of historical returns ──────────────────────────────
ax = axes[0]
max_lag = 24
hist_acf = [pd.Series(hist_dlogS).autocorr(lag=i) for i in range(1, max_lag+1)]
sim_acf  = [pd.Series(sim_dlogS).autocorr(lag=i)  for i in range(1, max_lag+1)]
lags     = range(1, max_lag+1)

ax.bar(lags, hist_acf, alpha=0.6, color="#1976D2", label="Historical")
ax.plot(lags, sim_acf, color="#E53935", marker="o",
        markersize=4, lw=1.5, label="Log-OU simulated")
conf = 1.96 / np.sqrt(len(hist_dlogS))
ax.axhline(conf,  color="gray", ls="--", lw=1, label="95% CI bounds")
ax.axhline(-conf, color="gray", ls="--", lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Autocorrelation")
ax.set_title("ACF of Log-Spread Returns\nHistorical vs Simulated")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Middle: Lag-1 scatter (mean reversion check) ──────────────────
ax = axes[1]
ax.scatter(hist_logS, hist_dlogS, alpha=0.3, s=8,
           color="#1976D2", label="Historical")
valid = np.isfinite(hist_logS) & np.isfinite(hist_dlogS)
m, b, r, p, _ = stats.linregress(hist_logS[valid], hist_dlogS[valid])
x_line = np.linspace(hist_logS.min(), hist_logS.max(), 100)
ax.plot(x_line, m*x_line + b, color="#E53935", lw=2,
        label=f"OLS fit (slope={m:.3f})")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("log S(t)")
ax.set_ylabel("Δlog S(t)")
ax.set_title(f"Mean Reversion: Δlog S vs log S\n"
             f"Slope={m:.3f}, p={p:.4f}, R²={r**2:.3f}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

# ── Right: Ljung-Box test ─────────────────────────────────────────
ax = axes[2]
from scipy.stats import chi2
lb_stats = []
lb_pvals = []
for lag in range(1, max_lag+1):
    n     = len(hist_dlogS)
    acfs  = [pd.Series(hist_dlogS).autocorr(lag=k) for k in range(1, lag+1)]
    Q     = n*(n+2) * sum(a**2/(n-k) for k, a in enumerate(acfs, 1))
    p_val = 1 - chi2.cdf(Q, lag)
    lb_stats.append(Q)
    lb_pvals.append(p_val)

ax.plot(range(1, max_lag+1), lb_pvals, color="#1976D2",
        marker="o", markersize=4, lw=1.5, label="Ljung-Box p-value")
ax.axhline(0.05, color="red", ls="--", lw=1.5,
           label="5% significance level")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("p-value")
ax.set_title("Ljung-Box Test for Autocorrelation\nH0: No autocorrelation")
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("../figures/experimental_autocorr_validation.png", dpi=150, bbox_inches="tight")
plt.show()

# Print mean reversion interpretation
print("Mean Reversion Validation:")
print(f"  OLS slope on Δlog S vs log S : {m:.4f}")
print(f"  p-value                       : {p:.4f}")
print(f"  Interpretation                : {'Mean reversion confirmed (slope < 0, p < 0.05)' if m < 0 and p < 0.05 else 'Weak or no mean reversion'}")
print(f"  Implied annual theta          : {-m * 12:.3f}")
print(f"  Model theta (normal regime)   : {CALIB['normal']['theta']:.3f}")

## CELL 18 — PD & Rating Chart


In [ ]:
# CELL 18

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("CPDO NAV-event estimates and illustrative thresholds (not a credit rating)", 
             fontsize=13, fontweight="bold")

# ── LEFT: PD vs Rating Scale ──────────────────────────────────────
ax = axes[0]

ratings    = list(RATING_SCALE.keys())
thresholds = [v * 100 for v in RATING_SCALE.values()]
bar_clr    = ["#1565C0","#1976D2","#1E88E5","#FFA000",
              "#F57C00","#E53935","#B71C1C"]

bars = ax.barh(ratings, thresholds, color=bar_clr, alpha=0.75, height=0.6)
if assigned in ratings:
    bars[ratings.index(assigned)].set_edgecolor("black")
    bars[ratings.index(assigned)].set_linewidth(2.5)

ax.axvline(pd_combined * 100, color="black", lw=2.5,
           label=f"CPDO PD = {pd_combined*100:.2f}%")
ax.axvspan(pd_lo * 100, pd_hi * 100, alpha=0.15, color="black",
           label="95% CI")
ax.axvline(RATING_SCALE["Aaa"] * 100, color="gold", lw=2, ls="--",
           label="Aaa threshold")

ax.set_xscale("log")
ax.invert_yaxis()
ax.set_xlabel("5yr PD (%, log scale)")
ax.set_title("Modelled event frequency vs illustrative thresholds (not a rating)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15, axis="x")

# Annotate assigned rating
ax.text(pd_combined * 100 * 1.1, ratings.index(assigned),
        f"  {assigned} / {SP_EQ.get(assigned,'--')}\n  ({pd_combined*100:.2f}%)",
        fontsize=9, va="center", fontweight="bold")

# ── RIGHT: Final NAV Distribution ────────────────────────────────
ax = axes[1]

final = mc["final_navs"] / 1e6  # in $M

ax.hist(final, bins=80, density=True, color="#1976D2",
        alpha=0.7, edgecolor="white", label="Final NAV distribution")

# Par line
ax.axvline(FACE_VALUE / 1e6, color="black", lw=2, ls="--",
           label=f"Par = $1.00M")

# Mean and median
ax.axvline(final.mean(), color="#FF9800", lw=2,
           label=f"Mean = ${final.mean():.3f}M")
ax.axvline(np.median(final), color="#43A047", lw=2, ls=":",
           label=f"Median = ${np.median(final):.3f}M")

# Shade shortfall region
ax.axvspan(0, FACE_VALUE / 1e6, alpha=0.08, color="red",
           label=f"Shortfall region ({pd_maturity:.1%} of paths)")

# 5th percentile
p5_nav = np.percentile(final, 5)
ax.axvline(p5_nav, color="red", lw=1.5, ls="--",
           label=f"5th pct = ${p5_nav:.3f}M")

ax.set_xlabel("Final NAV at Maturity ($M)")
ax.set_ylabel("Density")
ax.set_title("Distribution of Final NAV at Maturity (Month 120)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("../figures/experimental_pd_rating.png", dpi=150, bbox_inches="tight")
plt.show()

## CELL 19 — Spread Scalar Sensitvity


In [ ]:
# CELL 19 

def build_spread_scalar(treasury, cdx_raw, dbaa_raw, baa_scalar):
    treasury_m = to_month_end(treasury, ["rf_pct"])
    cdx_m      = to_month_end(cdx_raw,  ["cdx_bps"])
    dbaa_m     = to_month_end(dbaa_raw, ["baa_yield_pct"])
    m = treasury_m.merge(cdx_m, on="date", how="left")
    m = m.merge(dbaa_m, on="date", how="left")
    m["baa_proxy_bps"] = np.where(
        (m["date"] >= "2007-01-01") & (m["date"] <= "2016-12-31") &
        m["baa_yield_pct"].notna() & m["rf_pct"].notna(),
        (m["baa_yield_pct"] - m["rf_pct"]) * 100 * baa_scalar, np.nan)
    m["cdx_actual_bps"] = np.where(
        m["date"] >= "2020-01-01", m["cdx_bps"], np.nan)
    m["cdx_unified_bps"] = np.where(
        m["cdx_actual_bps"].notna(), m["cdx_actual_bps"],
        np.where(m["baa_proxy_bps"].notna(), m["baa_proxy_bps"], np.nan))
    m["source"] = np.where(
        m["cdx_actual_bps"].notna(), "CDX_actual",
        np.where(m["baa_proxy_bps"].notna(), "BAA_proxy", "missing"))
    return m.dropna(subset=["cdx_unified_bps"]).reset_index(drop=True)

SCALARS   = [0.35, 0.40, 0.50, 0.55]
FIXED_CPN = 20
N_SENS    = 5_000

print(f"BAA Scalar Sensitivity — Fixed Coupon = {FIXED_CPN} bps\n")
print(f"{'Scenario':<30}  {'PD 10yr':>9}  {'Annual PD':>10}  "
      f"{'E[NAV]':>12}  {'Implied':>9} ")
print("-" * 90)

for scalar in SCALARS:
    aligned_s = build_spread_scalar(treasury, cdx_raw, dbaa_raw, scalar)
    df_old_s  = aligned_s[
        (aligned_s["date"] >= "2007-01-01") &
        (aligned_s["date"] <= "2016-12-31") &
        (aligned_s["source"] == "BAA_proxy")].copy().reset_index(drop=True)
    df_rec_s  = aligned_s[
        (aligned_s["date"] >= "2020-01-01") &
        (aligned_s["source"] == "CDX_actual")].copy().reset_index(drop=True)

    reg_rec_s, _, _ = detect_regimes(df_rec_s)
    cf_rec = reg_rec_s["is_crisis"].values

    reg_old_s, _, _ = detect_regimes(df_old_s)
    cf_old = reg_old_s["is_crisis"].values
    par_rec = fit_log_ou_regime(df_rec_s["cdx_unified_bps"],
                                cf_rec, dt_obs=1/12, label="")
    par_old = fit_log_ou_regime(df_old_s["cdx_unified_bps"],
                                cf_old, dt_obs=1/12, label="")
    calib_s = weighted_params(par_rec, par_old, W_RECENT, W_OLD)

    _CALIB_orig = CALIB
    CALIB       = calib_s
    r           = run_mc(FIXED_CPN, n_paths=N_SENS,
                         verbose=False, seed=SWEEP_SEED)
    CALIB       = _CALIB_orig

    pd_10yr   = r["pcmb"]
    annual_pd = 1 - (1 - pd_10yr) ** (1/10)
    rating    = assign_rating(pd_10yr)
    scenario = f"Scalar = {scalar:.2f}{marker}"

    print(f"{scenario:<30}  {pd_10yr:>9.2%}  {annual_pd:>10.4%}  "
          f"${r['enav']:>10,.0f}  {rating:>9}")

print("-" * 90)

## CELL 20 — Leverage Cap Sensitivity

In [ ]:
# CELL 20 -- Leverage Cap 

import time

LEV_CAP_VALUES = [3, 5, 8, 10, 12, 15, 20]
FIXED_CPN      = 20
N_SENS         = 5_000

pd_sweep  = []
co_sweep  = []
mat_sweep = []
nav_sweep = []
rat_sweep = []

t0 = time.time()
for cap in LEV_CAP_VALUES:
    _orig_levcap = LEV_CAP
    LEV_CAP      = cap

    r = run_mc(FIXED_CPN, n_paths=N_SENS, verbose=False, seed=42)

    LEV_CAP = _orig_levcap

    pd_sweep.append(r["pcmb"]  * 100)
    co_sweep.append(r["pco"]   * 100)
    mat_sweep.append(r["pmat"] * 100)
    nav_sweep.append(r["enav"])
    rat_sweep.append(assign_rating(r["pcmb"]))

print(f"Runtime: {time.time()-t0:.1f}s\n")

# ── Table ─────────────────────────────────────────────────────────
print(f"Leverage Cap Sweep — Fixed Coupon = {FIXED_CPN} bps\n")
print(f"{'Cap':>6}  {'PD 10yr':>9}  {'P(Cash-out)':>12}  "
      f"{'P(Mat Short)':>13}  {'E[NAV]':>12}  {'Rating':>8}")
print("-" * 68)
for cap, pd, co, mat, nav, rat in zip(LEV_CAP_VALUES, pd_sweep, co_sweep,
                                       mat_sweep, nav_sweep, rat_sweep):
    marker = "  ◄ current" if cap == 15 else ""
    print(f"{cap:>5}x  {pd:>9.2f}%  {co:>12.2f}%  "
          f"{mat:>13.2f}%  ${nav:>10,.0f}  {rat:>8}{marker}")
print("-" * 68)

# ── Chart ─────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(LEV_CAP_VALUES, pd_sweep,  color="#E53935", marker="o",
         lw=2, markersize=6, label="P(combined) %")
ax1.plot(LEV_CAP_VALUES, mat_sweep, color="#FF9800", marker="^",
         lw=2, markersize=6, ls="--", label="P(mat shortfall) %")
ax1.set_xlabel("Max Leverage Cap (x)")
ax1.set_ylabel("PD (%)", color="#E53935")
ax1.tick_params(axis="y", labelcolor="#E53935")
ax1.set_ylim(0, max(max(pd_sweep) * 1.3, 5))

ax2 = ax1.twinx()
ax2.plot(LEV_CAP_VALUES, co_sweep, color="#43A047", marker="s",
         lw=2, markersize=6, label="P(cash-out) %")
ax2.set_ylabel("P(Cash-out) %", color="#43A047")
ax2.tick_params(axis="y", labelcolor="#43A047")
ax2.set_ylim(0, 105)

ax1.axvline(15, color="gray", ls=":", lw=1.5,
            label="Current cap = 15x")
ax1.axhline(BA_10YR_THRESHOLD * 100, color="black", ls="--",
            lw=1.2, label=f"Ba threshold ({BA_10YR_THRESHOLD:.0%})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           frameon=False, fontsize=9, loc="upper right")

ax1.set_title("PD and Cash-out Rate vs Max Leverage Cap\n"
              f"Fixed Coupon = {FIXED_CPN} bps | 5,000 paths per point",
              fontsize=11, fontweight="bold")
ax1.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig("../figures/experimental_leverage_sweep.png", dpi=150, bbox_inches="tight")
plt.show()